# Platoon Splits and Situational Analysis

**Objective:** Determine how each pitcher performs against LHB vs RHB, through the batting order, and in high-leverage situations.

**Key Question:** Do these pitchers have exploitable weaknesses that limit their effectiveness?

In [ ]:
from analysis_utils import *
set_style()
dfs = load_all()
COLORS = {'Ohtani':'#1f77b4','Sanchez':'#ff7f0e','Misiorowski':'#2ca02c'}

# ---- Platoon Splits ----
print("PLATOON SPLITS\n" + "=" * 110)
rows = []
for name, df in dfs.items():
    ps = platoon_splits(df)
    for side, d in ps.items():
        rows.append({'Pitcher':name,'vs':side,'PA':d['pa'],
                     'K%':f"{d['k_pct']:.1f}%",'BB%':f"{d['bb_pct']:.1f}%",
                     'AVG':f"{d['avg']:.3f}",'OBP':f"{d['obp']:.3f}",
                     'SLG':f"{d['slg']:.3f}",'OPS':f"{d['ops']:.3f}",'HR':d['hr']})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ---- Platoon OPS Visualization ----
fig, ax = plt.subplots(figsize=(10, 6))
names = list(dfs.keys())
x = np.arange(len(names)); w = 0.35
ps_all = {name: platoon_splits(df) for name, df in dfs.items()}
for i, name in enumerate(names):
    ps = ps_all[name]
    lhb = ps.get('L',{}).get('ops',0); rhb = ps.get('R',{}).get('ops',0)
    ax.bar(i-w/2, lhb, w, label='vs LHB' if i==0 else '', color='#d62728', alpha=0.85)
    ax.bar(i+w/2, rhb, w, label='vs RHB' if i==0 else '', color='#1f77b4', alpha=0.85)
    ax.text(i-w/2, lhb+0.015, f'{lhb:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i+w/2, rhb+0.015, f'{rhb:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("OPS Allowed")
ax.set_title("Platoon Splits: OPS Allowed vs LHB/RHB", fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('figures/platoon_ops.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Time Through Order ----
print("PERFORMANCE BY TIME THROUGH ORDER\n" + "=" * 90)
tto_col = 'n_thruorder_pitcher'
if tto_col in dfs['Ohtani'].columns:
    for name, df in dfs.items():
        print(f"\n{name}:")
        for tto in sorted(df[tto_col].dropna().unique()):
            sub = df[df[tto_col] == tto]
            pa = sub[sub['events'].notna() & (sub['events'] != '')]
            if len(pa) == 0: continue
            k = len(pa[pa['events']=='strikeout'])
            bb = len(pa[pa['events']=='walk'])
            hits = sum(len(pa[pa['events']==e]) for e in ['single','double','triple','home_run'])
            print(f"  TTO {int(tto):1d}: {len(pa):3d} PA  K%={k/len(pa)*100:.1f}%  BB%={bb/len(pa)*100:.1f}%  AVG={hits/len(pa):.3f}")